In [ ]:
"""
Plan_B / Experiment 1 — CCAL Multi-Seed Robustness Run.

Re-runs the honest (sequential 70/20/10) CCAL training across 5 seeds.
Reports mean ± std on overall accuracy, macro F1, and per-class F1.
This kills the 'single-seed' limitation noted in thesis Chapter 5.

Reuses CCAL training code verbatim from Plan_B/_shared/ccal_honest.py
(no edits to the original code in /home/metal/Experiment/related_work/).

Run on Kaggle T4 (same env as the thesis runs). Output CSV is saved to /kaggle/working.
"""
import os, sys, csv, gc, shutil, time, copy

# Make sibling _shared/ importable
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)
import wandb

from _shared.ccal_honest import (
    config, set_seed, prepare_sequential_data, CCALModel,
    train_model, load_best_model
)

# ===================== EXPERIMENT CONFIG =====================
SEEDS = [0, 1, 42, 123, 2024]
OUT_CSV = '/kaggle/working/ccal_multiseed_results.csv'
CKPT_DIR = '/kaggle/working/ccal_ckpts'
os.makedirs(CKPT_DIR, exist_ok=True)

CLASS_NAMES = ['Benign', 'Malignant', 'Normal']


def per_class_eval(model, loader, device):
    """Run model on loader, return overall + per-class metrics dict."""
    model.eval()
    preds, labels, probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            out, _ = model(x)
            p = F.softmax(out, dim=1)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y.cpu().numpy())
            probs.extend(p.cpu().numpy())

    acc = accuracy_score(labels, preds)
    prec_w = precision_score(labels, preds, average='weighted', zero_division=0)
    rec_w = recall_score(labels, preds, average='weighted', zero_division=0)
    f1_w = f1_score(labels, preds, average='weighted', zero_division=0)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(labels, np.array(probs), multi_class='ovr', average='weighted')
    except Exception:
        auc = float('nan')
    pc_f1 = f1_score(labels, preds, average=None, zero_division=0)

    return {
        'accuracy': acc, 'precision_w': prec_w, 'recall_w': rec_w,
        'f1_weighted': f1_w, 'f1_macro': f1_macro, 'auc': auc,
        'f1_benign': pc_f1[0], 'f1_malignant': pc_f1[1], 'f1_normal': pc_f1[2],
    }


def main():
    # W&B login (Kaggle secret)
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("wandb_api_key"))

    rows = []

    for seed in SEEDS:
        print(f"\n{'#'*70}\n# CCAL MULTI-SEED RUN — seed={seed}\n{'#'*70}")
        seed_t0 = time.time()
        set_seed(seed)

        # New W&B run per seed
        wandb.init(
            project="thesis_plan_b",
            name=f"ccal_multiseed_seed{seed}",
            config={**config, 'seed': seed, 'experiment': 'multiseed', 'arch': 'CCAL'},
            tags=['plan_b', 'multiseed', 'ccal', f'seed_{seed}'],
            reinit=True,
        )

        train_ld, val_ld, test_ld = prepare_sequential_data(
            config['data_dir'],
            train_ratio=config['train_ratio'],
            val_ratio=config['val_ratio'],
            target_total=config['target_total_images'],
            batch_size=config['batch_size'],
            image_size=config['image_size'],
        )

        model = CCALModel(num_classes=3,
                          num_heads=config['num_heads'],
                          key_dim=config['key_dim'])

        train_model(model, train_ld, val_ld,
                    num_epochs=config['epochs'],
                    learning_rate=config['learning_rate'],
                    device=config['device'])

        # Load best (saved as 'best_model.pth' by train_model)
        model = load_best_model(model, device=config['device'])
        m = per_class_eval(model, test_ld, config['device'])

        # Archive checkpoint per seed
        try:
            shutil.move('best_model.pth', f'{CKPT_DIR}/ccal_best_seed{seed}.pth')
        except FileNotFoundError:
            pass

        elapsed = time.time() - seed_t0
        row = {'arch': 'CCAL', 'seed': seed, 'time_sec': round(elapsed, 1), **m}
        rows.append(row)

        wandb.log({f'final/{k}': v for k, v in m.items()})
        wandb.finish()

        del model, train_ld, val_ld, test_ld
        torch.cuda.empty_cache()
        gc.collect()

        print(f"\n[CCAL seed={seed}] acc={m['accuracy']*100:.2f}%  "
              f"f1_macro={m['f1_macro']*100:.2f}  benign_f1={m['f1_benign']*100:.2f}  "
              f"({elapsed:.1f}s)")

    # Write CSV
    with open(OUT_CSV, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    # Quick summary
    accs = [r['accuracy'] for r in rows]
    benign = [r['f1_benign'] for r in rows]
    print(f"\n{'='*70}\nCCAL MULTI-SEED SUMMARY ({len(rows)} seeds)\n{'='*70}")
    print(f"  accuracy:  mean={np.mean(accs)*100:.2f}%  std={np.std(accs, ddof=1)*100:.2f}%")
    print(f"  benign F1: mean={np.mean(benign)*100:.2f}%  std={np.std(benign, ddof=1)*100:.2f}%")
    print(f"  CSV: {OUT_CSV}")


if __name__ == '__main__':
    main()
